# importing libraries

In [1]:
from collections import defaultdict
import math
import sys
import numpy as np 
import re
import glob
import pandas as pd
from nltk.stem import PorterStemmer
from functools import reduce
from nltk.corpus import stopwords

# Getting data and tokenization and removing stop words


In [2]:
docNames= []
document_filenames = {}
for name in glob.glob(r'Files'+'\*'):
        docNames.append(name)
N = len(docNames)
for i in range(N):
    document_filenames[i] = docNames[i]
dictionary = set()
docsLen=[]
tfIdfMatrix=defaultdict(dict)
idf = defaultdict(dict)
postings = defaultdict(dict)
posIndex= defaultdict(dict)
documents_content= defaultdict(dict)
document_frequency = defaultdict(int)
length = defaultdict(float)



In [3]:
def initialize_terms_and_postings():
    global dictionary, postings, docsLen, posIndex
    for id in document_filenames:
        f = open(document_filenames[id],'r')
        document = f.read().replace('\n'," ")
        documents_content[document_filenames[id].split('\\')[1]]=document
        f.close()
        terms = tokenize(document)
        unique_terms = set(terms)
        docsLen.append(len(unique_terms))
        dictionary = dictionary.union(unique_terms)
        #here we get the log tf weighting not just term frequency
        for term in unique_terms:
            postings[term][id] = 1+np.log10(int(terms.count(term)))
        #now we initialize postitional index
        poslst=[] #this is a position list for a term
        for term in terms:
            for i in range(0,len(terms)):
                if term == terms[i]:
                     poslst.append(i)
            posIndex[term][id] = poslst #we save positions in the positional index for each term in each doc 
            poslst=[] 


In [4]:
def tokenize(document):
        stop_words = stopwords.words('english')
        document = re.sub(" "," ", document)
        document = re.sub(r'won’t', 'will not', document)
        document = re.sub(r'can\’t', 'can not', document)
        document = re.sub(r'n\’t', 'not', document)
        document = re.sub(r'\’re', 'are', document)
        document = re.sub(r'\’s', 'is', document)
        document = re.sub(r'\’d', 'would', document)
        document = re.sub(r'\’ll', 'will', document)
        document = re.sub(r'\’t', 'not', document)
        document = re.sub(r'\’ve', ' have', document)
        document = re.sub(r'\’m', 'am', document)
        document = re.sub(r'[0–9]+', "", document)
        document = re.sub(r'[^\w\s]'," ", document)
        document = document.lower()
        terms=[]
        for word in document.split(" "):
            if word!="":
                if word not in stop_words:
                    terms.append(word)
        terms2=[]
        ps= PorterStemmer()
        for term in terms :
            term = ps.stem(term)
            terms2.append(term)            
        doc = list(filter(None, terms2)) 
        return doc
   

In [5]:
 # Document Frequency

# number of documents that include the term >  len(postings[term])
def initialize_document_frequencies():
    global document_frequency
    for term in dictionary:
        document_frequency[term] = len(postings[term])
        

In [6]:
initialize_terms_and_postings()
initialize_document_frequencies()


In [7]:
documents_content


defaultdict(dict,
            {'france.txt': 'New york times',
             'IR.txt': 'new york post york',
             'japan.txt': 'los angles times',
             'John.txt': 'John is quicker than Mary',
             'Mary.txt': 'Mary is quicker than John'})

# Positional Index

In [8]:
x = pd.DataFrame.from_dict(posIndex)
print(x)
y = pd.DataFrame.from_dict(document_frequency, orient='index')
print(y)

   new    york time post   lo angl john quicker mari
0  [0]     [1]  [2]  NaN  NaN  NaN  NaN     NaN  NaN
1  [0]  [1, 3]  NaN  [2]  NaN  NaN  NaN     NaN  NaN
2  NaN     NaN  [2]  NaN  [0]  [1]  NaN     NaN  NaN
3  NaN     NaN  NaN  NaN  NaN  NaN  [0]     [1]  [2]
4  NaN     NaN  NaN  NaN  NaN  NaN  [2]     [1]  [0]
         0
lo       1
john     2
time     2
new      2
angl     1
quicker  2
mari     2
post     1
york     2


# Log Term Frequency Weighting 

In [9]:
z = pd.DataFrame.from_dict(postings).replace(np.nan, 0)
print(z)

   new     york  time  post   lo  angl  mari  john  quicker
0  1.0  1.00000   1.0   0.0  0.0   0.0   0.0   0.0      0.0
1  1.0  1.30103   0.0   1.0  0.0   0.0   0.0   0.0      0.0
2  0.0  0.00000   1.0   0.0  1.0   1.0   0.0   0.0      0.0
3  0.0  0.00000   0.0   0.0  0.0   0.0   1.0   1.0      1.0
4  0.0  0.00000   0.0   0.0  0.0   0.0   1.0   1.0      1.0


# Inverse Document Frequency

In [10]:
def inverse_document_frequency(term):
    if term in dictionary:
        return np.log10(N/document_frequency[term])
    else:
        return 0.0

In [11]:
for term in dictionary: 
      idf[term]=float(inverse_document_frequency(term))
idfFrame = pd.DataFrame.from_dict(idf, orient='index', columns=['idf'])
print(idfFrame)

             idf
lo       0.69897
john     0.39794
time     0.39794
new      0.39794
angl     0.69897
quicker  0.39794
mari     0.39794
post     0.69897
york     0.39794


# TF-IDF Matrix for Collection


In [12]:
global tfIdfMatrix
for i in range(N):
    for term in postings:
            if i in postings[term]:
                tfIdfMatrix[term][i]= postings[term][i]* idf[term]
            else:  tfIdfMatrix[term][i]=0

In [13]:
matrix = pd.DataFrame.from_dict(tfIdfMatrix)
print(matrix)


       new      york     time     post       lo     angl     mari     john  \
0  0.39794  0.397940  0.39794  0.00000  0.00000  0.00000  0.00000  0.00000   
1  0.39794  0.517732  0.00000  0.69897  0.00000  0.00000  0.00000  0.00000   
2  0.00000  0.000000  0.39794  0.00000  0.69897  0.69897  0.00000  0.00000   
3  0.00000  0.000000  0.00000  0.00000  0.00000  0.00000  0.39794  0.39794   
4  0.00000  0.000000  0.00000  0.00000  0.00000  0.00000  0.39794  0.39794   

   quicker  
0  0.00000  
1  0.00000  
2  0.00000  
3  0.39794  
4  0.39794  


# Norm of query and documents

In [14]:
def norm():
    for id in document_filenames:
        l = 0
        for term in dictionary:
            l += tfIdfMatrix[term][id]**2
        length[id] = math.sqrt(l)
        
def qnorm(query):
    QueryTfIdf = queryPreperation(query)
    l2 = 0
    for term in QueryTfIdf.values():
        if type(term)== float: l2 += term**2
        qlength = math.sqrt(l2)
    return qlength
norm()
length

defaultdict(float,
            {0: 0.6892523133843688,
             1: 0.9565362643747136,
             2: 1.0655864040165348,
             3: 0.6892523133843688,
             4: 0.6892523133843688})

# Query Preperation

In [15]:
#this is to make a tfidf for the query
QueryFreqList = defaultdict(dict)
QueryTfIdfMatrix = defaultdict(dict)
queryLen= None
def queryPreperation(query):
    terms = tokenize(query)
    unique_terms = set(terms)
    queryList=[]
    for term in unique_terms: 
        if term in dictionary: 
            queryList.append(term)
    dictionary1 = dictionary.union(queryList)
    for term in dictionary1:
        if int(terms.count(term)) != 0:
             QueryFreqList[term]= 1+np.log10(int(terms.count(term)))
        else: QueryFreqList[term]=0
    for i in range(N):
        for term in QueryFreqList.keys():
                if QueryFreqList[term]!=0: 
                    QueryTfIdfMatrix[term]= float(QueryFreqList[term]* idf[term])
                else:  QueryTfIdfMatrix[term]=0
    return QueryTfIdfMatrix


In [16]:
#using the positional index to check if the terms in the right order in documents
def phraseQuery(query):
    positionsDict = {}
    listOflsts = []
    listOfRelDocId = []
    terms = tokenize(query)
    for id in document_filenames:
        for term in terms:
            if term in dictionary:
                if id in posIndex[term]:
                    listOflsts.append(posIndex[term][id])
        positionsDict[id] = listOflsts
        listOflsts = []
    empty_keys = [k for k,v in positionsDict.items() if v==[]]
    for k in empty_keys:
        del positionsDict[k]
    #getting the relevant docs
    for docId,listOfLists in positionsDict.items():
        listOfLists2 = [] #this is list of position lists after subtraction from i 
        i = 0
        for lst in listOfLists:
            lst2 = [] #this is positions list after subtraction from i 
            for item in lst:
                lst2.append(item - i)
            listOfLists2.append(lst2)
            i += 1
        # now we get intersection of lists
        # if not empty, they're in order
        # if empty they're not in order
        result = list(reduce(set.intersection, [set(item) for item in listOfLists2]))
        if result: 
            listOfRelDocId.append(docId)
    return listOfRelDocId

# Cosine Similarity 

In [17]:
def similarity(query,id):
    similarity = 0.0
    qlen = qnorm(query)
    QueryTfIdf = queryPreperation(query)
    query = tokenize(query)
    for term in query:
        if term in dictionary:
                similarity += (QueryTfIdf[term]/qlen)*(tfIdfMatrix[term][id]/length[id])
    if similarity > 1: similarity = 1.00000
    return similarity


# Search

In [18]:
def do_search(query):
    filenames = {}
    relDoc= []
    tPositions={}
    listOfPositions=[]
    scores = {}
    dataframe = defaultdict(dict)
    relDocs = phraseQuery(query)
    for i in relDocs:
        scores[i] = similarity(query,i)
    for i in relDocs: 
        if scores[i]!= 0:
             dataframe[document_filenames[i].split('\\')[1]] = scores[i]
    df = pd.DataFrame.from_dict(dataframe,orient='index',columns = ['Cosine Similarity(Query,Document)'])
    df.sort_values(by=['Cosine Similarity(Query,Document)'], inplace=True, ascending = False)
    if dataframe == {}: return 0 
    else: return dataframe
    

# similarity between query and each document

In [ ]:
while True:
        query = input("Search query >> ")
        scores =  do_search(query)
        if scores != 0:
            for fileName, score in scores.items():
                print(str(fileName)+' : '+str(score))
                print(documents_content[fileName])
        else: print(" ***there's no relevant docs***")

Search query >> new york
france.txt : 0.816496580927726
New york times
IR.txt : 0.6768983377020721
new york post york
Search query >> no such terms
 ***there's no relevant docs***
Search query >> is quicker
John.txt : 0.5773502691896257
John is quicker than Mary
Mary.txt : 0.5773502691896257
Mary is quicker than John
Search query >> john mary
 ***there's no relevant docs***
Search query >> john is quicker
John.txt : 0.816496580927726
John is quicker than Mary
Search query >> los angles
japan.txt : 0.9276515316806165
los angles times
